# 3주차 실습 (1) - Information Theory from scratch

> 겁나 보이지만, 정보이론은 **"하나의 아이디어가 옷만 갈아입는" 것**이다.
> 벽돌은 딱 하나: **`-log(확률) = 놀람(surprise)`**.
> 그 위에 사다리를 한 칸씩 쌓는다 → 정보량 → 엔트로피 → 크로스엔트로피 → KL.

## 1️⃣ 정보량 = "놀람의 크기"

핵심 질문: **어떤 사건이 일어났을 때, 그게 얼마나 놀라운가?**

- 동전 앞면(50%) → 별로 안 놀람
- 로또 당첨(0.0000…%) → 엄청 놀람
- "내일 해가 뜬다"(100%) → 전혀 안 놀람 (놀람 = 0)

수식 한 줄: **`I(x) = -log(p)`**

`-log` 를 쓰는 이유만 알면 된다. 확률 `p` 가 작을수록(드물수록) `-log(p)` 는 커진다.
`p=1` 이면 `-log(1)=0` (안 놀람). 딱 직관과 맞는다.

> 💡 단위: log 밑이 2면 **bit**, e면 **nat**. "cm냐 inch냐" 차이. PyTorch는 nat.

In [6]:
import math


def information_content(p, base=2):
    if p <= 0 or p > 1:
        return float("inf") if p <= 0 else 0.0
    return -math.log(p) / math.log(base)


# 확률이 낮을수록(=드물수록) 놀람이 커진다
for p in [0.5, 1 / 6, 0.001, 1.0]:
    print(f"p={p:<6}  놀람 = {information_content(p):.2f} bits")

p=0.5     놀람 = 1.00 bits
p=0.16666666666666666  놀람 = 2.58 bits
p=0.001   놀람 = 9.97 bits
p=1.0     놀람 = -0.00 bits


## 2️⃣ 엔트로피 = "평균 놀람"

정보량은 사건 하나짜리. **엔트로피는 분포 전체의 평균 놀람**이다.

**`H(P) = -Σ p·log(p)`** ← 각 결과의 놀람을 그 확률로 가중평균.

아래 결과로 감을 잡자:
- **공정한 동전 = 1 bit** → 결과를 전혀 예측 못함 = 가장 불확실
- **편향 동전(99:1) = 0.08 bit** → 거의 앞면 나올 걸 아니까 놀랄 일이 적음
- **공정 주사위 = 2.58 bit** → 6갈래라 더 불확실

즉 **엔트로피 = 그 분포에 담긴 "불확실성의 양"**. 뻔하면 낮고, 예측 불가면 높다.
(= "이보다 더 짧게 압축 못 하는 한계선")

In [ ]:
def entropy(probs, base=2):
    return sum(p * information_content(p, base) for p in probs if p > 0)


print(f"공정한 동전:   {entropy([0.5, 0.5]):.4f} bits  (가장 불확실)")
print(f"편향 동전:     {entropy([0.99, 0.01]):.4f} bits  (거의 뻔함)")
print(f"공정 주사위:   {entropy([1 / 6] * 6):.4f} bits")

## 3️⃣ 크로스 엔트로피 = "틀린 예측으로 인한 놀람" ← 그 손실함수

여기가 핵심. 모델 훈련 때 쓰는 `CrossEntropyLoss()` 가 바로 이것.

상황: **진짜 정답 분포는 P인데, 내 모델은 Q라고 예측**했다.

**`H(P,Q) = -Σ p·log(q)`** ← 놀람 계산에 확률을 **모델 예측 q** 로 쓴 것.

직관: 모델(Q)이 진짜(P)와 비슷하면 놀랄 일이 적어 값이 작고, 엉터리면 계속 놀라 값이 커진다.

분류에선 정답이 원-핫(정답만 1)이라 식이 확 줄어든다:

**`H(P,Q) = -log(q[정답])`**

→ **"정답 클래스에 모델이 매긴 확률"에 -log.** 정답을 0.9로 예측하면 `-log(0.9)=0.1`(작은 손실),
0.1로 예측하면 `-log(0.1)=2.3`(큰 손실). **손실을 줄인다 = 정답 확률을 높인다.** 끝.

In [4]:
def cross_entropy(p, q, base=2):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            if qi <= 0:
                return float("inf")
            total += pi * (-math.log(qi) / math.log(base))
    return total


# 원-핫 정답(2번 클래스가 정답) vs 두 모델의 예측
true = [0.0, 0.0, 1.0]
good = [0.1, 0.1, 0.8]      # 정답에 0.8 -> 손실 작음
bad = [0.4, 0.4, 0.2]       # 정답에 0.2 -> 손실 큼
print(f"좋은 예측(정답 0.8): CE = {cross_entropy(true, good):.4f}")
print(f"나쁜 예측(정답 0.2): CE = {cross_entropy(true, bad):.4f}")
print(f"직접 확인: -log2(0.8) = {-math.log2(0.8):.4f}, -log2(0.2) = {-math.log2(0.2):.4f}")

좋은 예측(정답 0.8): CE = 0.3219
나쁜 예측(정답 0.2): CE = 2.3219
직접 확인: -log2(0.8) = 0.3219, -log2(0.2) = 2.3219


## 4️⃣ KL 발산 = "어쩔 수 없는 부분을 뺀 순수 오차"

크로스엔트로피를 쪼개면:

**`크로스엔트로피 H(P,Q) = 엔트로피 H(P) + KL발산 D_KL(P‖Q)`**

- `H(P)` = **진짜 데이터 자체의 불확실성** → 내가 어쩔 수 없는 고정값.
- `KL`  = **내 모델이 틀려서 생긴 추가 오차** → 이게 내가 줄일 수 있는 부분!

그래서 **크로스엔트로피 최소화 = KL 최소화 = 내 모델 분포를 진짜 분포에 밀착.**
훈련이 하는 일이 정확히 이거다. (아래에서 `CE == H + KL` 을 눈으로 확인)

> ⚠️ KL은 방향이 있다: `KL(P‖Q) ≠ KL(Q‖P)`. "거리"라 부르지만 진짜 거리는 아님.

In [5]:
def kl_divergence(p, q, base=2):
    return cross_entropy(p, q, base) - entropy(p, base)


p = [0.7, 0.2, 0.1]
for name, q in [("good", [0.6, 0.25, 0.15]), ("bad", [0.1, 0.1, 0.8])]:
    ce, h, kl = cross_entropy(p, q), entropy(p), kl_divergence(p, q)
    print(f"[{name}] CE={ce:.4f} = H={h:.4f}(어쩔수없음) + KL={kl:.4f}(내 오차)  "
          f"-> 합 {h + kl:.4f}")

[good] CE=1.1896 = H=1.1568(어쩔수없음) + KL=0.0328(내 오차)  -> 합 1.1896
[bad] CE=3.0219 = H=1.1568(어쩔수없음) + KL=1.8651(내 오차)  -> 합 3.0219


## 5️⃣ 보너스: 크로스 엔트로피 == 음의 로그우도(NLL)

"크로스엔트로피를 줄인다"는 "정답 데이터가 나올 확률(우도)을 최대화한다"와 **완전히 같은 말**이다.
`-Σ log(정답확률)` 이 곧 크로스엔트로피이기 때문. 1000개 샘플로 두 값이 같음을 확인.

In [7]:
import random


def softmax(logits):
    m = max(logits)
    exps = [math.exp(z - m) for z in logits]
    s = sum(exps)
    return [e / s for e in exps]


def ce_loss(true_class, logits):
    return -math.log(softmax(logits)[true_class])


random.seed(42)
n, k = 1000, 3
labels = [random.randint(0, k - 1) for _ in range(n)]
logits = [[random.gauss(0, 1) for _ in range(k)] for _ in range(n)]
ce = sum(ce_loss(y, z) for y, z in zip(labels, logits)) / n
nll = -sum(math.log(softmax(z)[y]) for y, z in zip(labels, logits)) / n
print(f"Cross-entropy = {ce:.6f}")
print(f"NLL           = {nll:.6f}")
print(f"차이          = {abs(ce - nll):.2e}  (사실상 0 = 같은 것)")

Cross-entropy = 1.400910
NLL           = 1.400910
차이          = 0.00e+00  (사실상 0 = 같은 것)


## 6️⃣ 나머지 조연 — 상호정보량 · 퍼플렉서티

- **상호정보량 `I(X;Y)`**: "X를 알면 Y의 불확실성이 얼마나 줄어드나". 독립이면 **0**.
  특징 선택에서 "이 feature가 정답과 관련 있나?" 잴 때 씀 (상관계수보다 강력, 비선형도 잡음).
- **퍼플렉서티 `PP = e^(크로스엔트로피)`**: "모델이 평균 몇 개 선택지에서 헷갈리나".
  언어모델 평가지표. 50이면 "매번 50개 중 찍는 수준", 낮을수록 좋음.

In [8]:
def mutual_information(joint, base=2):
    rows, cols = len(joint), len(joint[0])
    px = [sum(joint[i][j] for j in range(cols)) for i in range(rows)]
    py = [sum(joint[i][j] for i in range(rows)) for j in range(cols)]
    return sum(joint[i][j] * math.log(joint[i][j] / (px[i] * py[j])) / math.log(base)
               for i in range(rows) for j in range(cols) if joint[i][j] > 0)


print(f"독립(관련 없음):  MI = {mutual_information([[0.25, 0.25], [0.25, 0.25]]):.4f} bits")
print(f"의존(관련 있음):  MI = {mutual_information([[0.45, 0.05], [0.05, 0.45]]):.4f} bits")

# 퍼플렉서티: 크로스엔트로피의 지수
loss = ce_loss(0, [2.0, 1.0, 0.1])
print(f"\n예측 손실 {loss:.3f} nats -> 퍼플렉서티 {math.exp(loss):.2f} "
      f"(약 {math.exp(loss):.1f}개 중 헷갈리는 수준)")

독립(관련 없음):  MI = 0.0000 bits
의존(관련 있음):  MI = 0.5310 bits

예측 손실 0.417 nats -> 퍼플렉서티 1.52 (약 1.5개 중 헷갈리는 수준)


## 🎯 한 문장 정리

> **정보량 → (평균 내면) 엔트로피 → (예측으로 재면) 크로스엔트로피 → (엔트로피 빼면) KL.**
> 전부 `-log(확률) = 놀람` 이라는 **하나의 벽돌**로 쌓은 탑이다.

| 개념 | 한 줄 | 언제 |
| --- | --- | --- |
| 정보량 `-log p` | 사건 하나의 놀람 | 벽돌 |
| 엔트로피 `H(P)` | 분포의 평균 놀람 = 불확실성 | 데이터 자체 |
| 크로스엔트로피 `H(P,Q)` | 틀린 예측(Q)의 놀람 | **분류 손실** |
| KL `D_KL(P‖Q)` | CE에서 H를 뺀 순수 오차 | 훈련이 줄이는 것 |
| 퍼플렉서티 `e^H(P,Q)` | 헷갈리는 선택지 수 | LM 평가 |

개념 노트: [[🎲 엔트로피와 정보량]] · [[📏 크로스 엔트로피와 KL 발산]] · [[🔗 상호정보량]] (Obsidian LLM_Wiki)